# Data Generation: H-Net and SmilesPE Tokenization

This notebook generates tokenization data for all 6 trained H-Net models and the SmilesPE benchmark.

## Overview

**Trained Models:**
1. PI1M concat, 1 epoch (68M bytes): `run_large_20251113_181705`
2. PI1M no concat, 5 epoch (240M bytes): `run_large_20251111_075600`
3. PI1M concat, 5 epoch (240M bytes): `run_large_20251111_181836`
4. PI1M concat, 22 epoch (1B bytes): `run_large_20251112_150502`
5. MOSES no concat, 5 epoch (360M bytes): `run_large_20251113_074900`
6. MOSES concat, 5 epoch (360M bytes): `run_large_20251112_071557`

**Process:**
1. Load each model checkpoint
2. Run tokenization inference on full dataset
3. Save results efficiently
4. Generate statistics
5. Run SmilesPE benchmark on both datasets


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Setup plotting style
sns.set_style("whitegrid")
sns.set_context("talk")
plt.rcParams['figure.figsize'] = (12, 8)

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Import analysis utilities
from analysis.utils.inference import (
    load_model, run_tokenization_inference, 
    save_tokenization_results, get_model_info
)
from analysis.utils.statistics import compute_token_statistics, TokenStatistics
from analysis.utils.benchmark import SmilesPEBenchmark

print("Imports successful!")
print(f"Project root: {project_root}")


## 1. Define Model Configurations

Define all 6 trained models to analyze.


In [ ]:
# Define all models to analyze
models = [
    {
        'name': 'PI1M_concat_1epoch',
        'path': project_root / 'checkpoints' / 'run_large_20251113_181705',
        'description': 'PI1M with 10-PSMILES concatenation, 1 epoch (68M bytes)',
        'dataset': 'PI1M',
        'concatenate': True,
        'epochs': 1,
    },
    {
        'name': 'PI1M_noconcat_5epoch',
        'path': project_root / 'checkpoints' / 'run_large_20251111_075600',
        'description': 'PI1M no concatenation, 5 epoch (240M bytes)',
        'dataset': 'PI1M',
        'concatenate': False,
        'epochs': 5,
    },
    {
        'name': 'PI1M_concat_5epoch',
        'path': project_root / 'checkpoints' / 'run_large_20251111_181836',
        'description': 'PI1M with 10-PSMILES concatenation, 5 epoch (240M bytes)',
        'dataset': 'PI1M',
        'concatenate': True,
        'epochs': 5,
    },
    {
        'name': 'PI1M_concat_22epoch',
        'path': project_root / 'checkpoints' / 'run_large_20251112_150502',
        'description': 'PI1M with 10-PSMILES concatenation, 22 epoch (1B bytes)',
        'dataset': 'PI1M',
        'concatenate': True,
        'epochs': 22,
    },
    {
        'name': 'MOSES_noconcat_5epoch',
        'path': project_root / 'checkpoints' / 'run_large_20251113_074900',
        'description': 'MOSES no concatenation, 5 epoch (360M bytes)',
        'dataset': 'MOSES',
        'concatenate': False,
        'epochs': 5,
    },
    {
        'name': 'MOSES_concat_5epoch',
        'path': project_root / 'checkpoints' / 'run_large_20251112_071557',
        'description': 'MOSES with 10-SMILES concatenation, 5 epoch (360M bytes)',
        'dataset': 'MOSES',
        'concatenate': True,
        'epochs': 5,
    },
]

# Display model configurations
print("Models to analyze:")
for i, model in enumerate(models, 1):
    print(f"{i}. {model['name']}")
    print(f"   Description: {model['description']}")
    print(f"   Path: {model['path']}")
    print()


## 2. Run H-Net Tokenization Inference

**NOTE:** This section requires GPU. Skip if GPU is busy and load pre-generated results instead.

For each model, we'll:
1. Load the checkpoint
2. Run inference on the full dataset
3. Save tokenization results
4. Compute and save statistics


In [ ]:
# Set this to True when GPU is available to run inference
RUN_HNET_INFERENCE = False  # Set to True when ready

if RUN_HNET_INFERENCE:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    for model_config in models:
        print(f"\n{'='*60}")
        print(f"Processing: {model_config['name']}")
        print(f"{'='*60}")
        
        try:
            # Load model
            model, info = load_model(str(model_config['path']), device=device)
            
            # Run tokenization inference
            results = run_tokenization_inference(
                model=model,
                dataset_csv=info['dataset_csv'],
                dataset_type=info['dataset_type'],
                device=device,
                max_samples=None,  # Process all samples
            )
            
            # Save results
            output_path = project_root / 'analysis' / 'data' / 'hnet_results' / f"{model_config['name']}_tokenization.pkl"
            save_tokenization_results(results, str(output_path))
            
            # Compute statistics
            stats = compute_token_statistics(results)
            
            # Save statistics
            stats_path = project_root / 'analysis' / 'data' / 'statistics' / f"{model_config['name']}_stats.json"
            stats.save(str(stats_path))
            
            print(f"\nResults saved to: {output_path}")
            print(f"Statistics saved to: {stats_path}")
            print(f"\nSummary:")
            for key, value in stats.get_summary().items():
                print(f"  {key}: {value}")
            
            # Free GPU memory
            del model
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"Error processing {model_config['name']}: {e}")
            import traceback
            traceback.print_exc()
else:
    print("⚠️ H-Net inference is DISABLED. Set RUN_HNET_INFERENCE=True to run.")
    print("This is to avoid GPU memory errors while training is running.")


## 3. Run SmilesPE Benchmark

Run SmilesPE tokenization on both datasets (PI1M and MOSES).
This doesn't require GPU and can be run anytime.


In [ ]:
# Set to True to run SmilesPE tokenization
RUN_SMILESPЕ_BENCHMARK = True

if RUN_SMILESPЕ_BENCHMARK:
    # Initialize SmilesPE
    vocab_path = project_root / 'analysis' / 'data' / 'SPE_ChEMBL.txt'
    spe = SmilesPEBenchmark(str(vocab_path))
    
    # Define datasets
    datasets = [
        {
            'name': 'PI1M',
            'csv': project_root / 'datasets' / 'PI1M' / 'PI1M_v2.csv',
            'type': 'PI1M',
        },
        {
            'name': 'MOSES',
            'csv': project_root / 'datasets' / 'moses' / 'smiles-molecules-moses_all.csv',
            'type': 'MOSES',
        },
    ]
    
    # Process each dataset
    for dataset in datasets:
        print(f"\n{'='*60}")
        print(f"Processing SmilesPE for: {dataset['name']}")
        print(f"{'='*60}")
        
        try:
            # Run tokenization
            results = spe.tokenize_dataset(
                str(dataset['csv']),
                dataset['type'],
                max_samples=None,
            )
            
            # Save results
            output_path = project_root / 'analysis' / 'data' / 'smilesPE_results' / f"SmilesPE_{dataset['name']}_tokenization.pkl"
            spe.save_results(results, str(output_path))
            
            # Compute statistics
            stats = compute_token_statistics(results)
            
            # Save statistics
            stats_path = project_root / 'analysis' / 'data' / 'statistics' / f"SmilesPE_{dataset['name']}_stats.json"
            stats.save(str(stats_path))
            
            print(f"\nResults saved to: {output_path}")
            print(f"Statistics saved to: {stats_path}")
            print(f"\nSummary:")
            for key, value in stats.get_summary().items():
                print(f"  {key}: {value}")
                
        except Exception as e:
            print(f"Error processing {dataset['name']}: {e}")
            import traceback
            traceback.print_exc()
else:
    print("⚠️ SmilesPE benchmark is DISABLED. Set RUN_SMILESPЕ_BENCHMARK=True to run.")


## 4. Summary of Generated Data

Check what data has been generated and is available for analysis.


In [ ]:
import os

# Check H-Net results
hnet_results_dir = project_root / 'analysis' / 'data' / 'hnet_results'
print("H-Net Tokenization Results:")
print("-" * 50)
if hnet_results_dir.exists():
    files = sorted(hnet_results_dir.glob("*.pkl"))
    if files:
        for f in files:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"  ✓ {f.name} ({size_mb:.2f} MB)")
    else:
        print("  ⚠️ No files found")
else:
    print("  ⚠️ Directory doesn't exist")

# Check SmilesPE results
spe_results_dir = project_root / 'analysis' / 'data' / 'smilesPE_results'
print("\nSmilesPE Tokenization Results:")
print("-" * 50)
if spe_results_dir.exists():
    files = sorted(spe_results_dir.glob("*.pkl"))
    if files:
        for f in files:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"  ✓ {f.name} ({size_mb:.2f} MB)")
    else:
        print("  ⚠️ No files found")
else:
    print("  ⚠️ Directory doesn't exist")

# Check statistics
stats_dir = project_root / 'analysis' / 'data' / 'statistics'
print("\nStatistics Files:")
print("-" * 50)
if stats_dir.exists():
    files = sorted(stats_dir.glob("*.json"))
    if files:
        for f in files:
            print(f"  ✓ {f.name}")
    else:
        print("  ⚠️ No files found")
else:
    print("  ⚠️ Directory doesn't exist")

print("\n" + "="*50)
print("Data generation complete!")
print("Proceed to analysis notebooks for comparative studies.")
